# FLARE Colab Runner

Run this notebook in Google Colab (with a GPU Runtime) to evaluate FLARE and baselines. Note that you need to upload this entire repository to Google Drive or clone it directly in Colab before running.

In [ ]:
# 1. Mount Google Drive FIRST so we can save everything directly!
from google.colab import drive
drive.mount('/content/drive')

# 2. Enter the active-rag directory (fixes the 'No such file or directory' error)
%cd /content/active-rag

# 3. Install dependencies for Pyserini (Java 21) and Python packages
!apt-get update -qq
!apt-get install openjdk-21-jdk-headless -qq > /dev/null
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"

!pip install -r requirements.txt

## Download Datasets

This cell checks every dataset's `dev.json` and downloads only what is missing.
Safe to re-run: already-present files are never re-downloaded.

| Dataset | Source |
|---|---|
| `2wikimultihopqa` | `xanhho/2WikiMultihopQA` on HuggingFace |
| `strategyqa` | `wics/strategy-qa` on HuggingFace |
| `asqa` | `din0s/asqa` on HuggingFace |
| `wikiasp` | `neulab/WikiAsp` on HuggingFace |

In [ ]:
import os, json

try:
    import datasets as hf_datasets
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "datasets", "-q"], check=True)
    import datasets as hf_datasets

def save_json(records, path):
    with open(path, "w") as f:
        json.dump(records, f)
    print(f"  Saved {len(records)} examples -> {path}")

# ── 1. 2WikiMultiHopQA ────────────────────────────────────────────
dev_file = "data/2wikimultihopqa/dev.json"
if os.path.exists(dev_file):
    print(f"[SKIP] 2wikimultihopqa already present")
else:
    print("[DOWNLOAD] 2wikimultihopqa ...")
    os.makedirs("data/2wikimultihopqa", exist_ok=True)
    ds = hf_datasets.load_dataset("xanhho/2WikiMultihopQA", split="validation")
    records = [
        {"_id": r.get("id", ""), "question": r.get("question", ""), "answer": r.get("answer", ""),
         "type": r.get("type", ""), "context": r.get("context", []), "supporting_facts": r.get("supporting_facts", [])}
        for r in ds
    ]
    save_json(records, dev_file)

# ── 2. StrategyQA ─────────────────────────────────────────────────
# NOTE: wics/strategy-qa uses a deprecated .py script (datasets v3+ rejects it).
# We load ChilleD/StrategyQA instead (same data, native parquet). If that also
# fails for any reason we fall back to the raw wics parquet files directly.
dev_file = "data/strategyqa/dev.json"
if os.path.exists(dev_file):
    print(f"[SKIP] strategyqa already present")
else:
    print("[DOWNLOAD] strategyqa ...")
    os.makedirs("data/strategyqa", exist_ok=True)
    try:
        ds = hf_datasets.load_dataset("ChilleD/StrategyQA", split="test")
    except Exception as e1:
        print(f"  ChilleD/StrategyQA failed ({e1}), falling back to wics parquet ...")
        try:
            ds = hf_datasets.load_dataset(
                "parquet",
                data_files="hf://datasets/wics/strategy-qa/data/test-*.parquet",
                split="train",
            )
        except Exception as e2:
            raise RuntimeError(
                f"Both StrategyQA sources failed.\n  Primary: {e1}\n  Fallback: {e2}"
            ) from e2
    records = [
        {"qid": r.get("qid", r.get("id", "")), "question": r.get("question", ""),
         "answer": str(r.get("answer", "")).lower()}
        for r in ds
    ]
    save_json(records, dev_file)

# ── 3. ASQA ───────────────────────────────────────────────────────
dev_file = "data/asqa/dev.json"
if os.path.exists(dev_file):
    print(f"[SKIP] asqa already present")
else:
    print("[DOWNLOAD] asqa ...")
    os.makedirs("data/asqa", exist_ok=True)
    ds = hf_datasets.load_dataset("din0s/asqa", split="dev")
    records = [
        {"id": r.get("sample_id", r.get("id", "")),
         "ambiguous_question": r.get("ambiguous_question", r.get("question", "")),
         "answer": r.get("answer", "")}
        for r in ds
    ]
    save_json(records, dev_file)

# ── 4. WikiAsp ────────────────────────────────────────────────────
# NOTE: "neulab/WikiAsp" (capital A) no longer exists on the Hub.
# The correct id is "neulab/wiki_asp" with domain-specific subsets.
# We iterate over all 12 subsets and concatenate them.
dev_file = "data/wikiasp/dev.json"
if os.path.exists(dev_file):
    print(f"[SKIP] wikiasp already present")
else:
    print("[DOWNLOAD] wikiasp ...")
    os.makedirs("data/wikiasp", exist_ok=True)
    _WIKI_ASP_SUBSETS = [
        "album", "animal", "infrastructure", "mean_of_transportation",
        "office_holder", "plant", "single", "soccer_player",
        "software", "television_show", "town", "written_work",
    ]
    records = []
    for _subset in _WIKI_ASP_SUBSETS:
        _ds = hf_datasets.load_dataset("neulab/wiki_asp", _subset, split="validation")
        records.extend(
            {"id": r.get("exid", r.get("id", "")), "input": r.get("input", ""), "output": r.get("output", "")}
            for r in _ds
        )
        print(f"  {_subset}: {len(_ds)} examples")
    save_json(records, dev_file)

print("\nAll datasets ready.")

## Run Experiments

Because we added `--results_dir /content/drive/MyDrive/active-rag-results`, your outputs will automatically save directly to Google Drive as they generate! If you get disconnected, nothing is lost.

Each dataset x eval_mode pair gets its **own subdirectory** so stale `predictions.jsonl` files from prior runs never interfere.

In [ ]:
import os
os.environ["PYTHONPATH"] = ".:" + os.environ.get("PYTHONPATH", "")

# ── 2WikiMultiHopQA ───────────────────────────────────────────────
!python scripts/run_flare.py --eval_mode no_retrieval --dataset 2wikihop --data_path data/2wikimultihopqa --results_dir /content/drive/MyDrive/active-rag-results/2wiki_no_retrieval

In [ ]:
!python scripts/run_flare.py --eval_mode single_retrieval --dataset 2wikihop --data_path data/2wikimultihopqa --results_dir /content/drive/MyDrive/active-rag-results/2wiki_single_retrieval

In [ ]:
!python scripts/run_flare.py --eval_mode flare --dataset 2wikihop --data_path data/2wikimultihopqa --results_dir /content/drive/MyDrive/active-rag-results/2wiki_flare

In [ ]:
# ── StrategyQA ────────────────────────────────────────────────────
!python scripts/run_flare.py --eval_mode no_retrieval --dataset strategyqa --data_path data/strategyqa --results_dir /content/drive/MyDrive/active-rag-results/strategyqa_no_retrieval

In [ ]:
!python scripts/run_flare.py --eval_mode single_retrieval --dataset strategyqa --data_path data/strategyqa --results_dir /content/drive/MyDrive/active-rag-results/strategyqa_single_retrieval

In [ ]:
!python scripts/run_flare.py --eval_mode flare --dataset strategyqa --data_path data/strategyqa --results_dir /content/drive/MyDrive/active-rag-results/strategyqa_flare

In [ ]:
# ── ASQA ──────────────────────────────────────────────────────────
!python scripts/run_flare.py --eval_mode no_retrieval --dataset asqa --data_path data/asqa --results_dir /content/drive/MyDrive/active-rag-results/asqa_no_retrieval

In [ ]:
!python scripts/run_flare.py --eval_mode single_retrieval --dataset asqa --data_path data/asqa --results_dir /content/drive/MyDrive/active-rag-results/asqa_single_retrieval

In [ ]:
!python scripts/run_flare.py --eval_mode flare --dataset asqa --data_path data/asqa --results_dir /content/drive/MyDrive/active-rag-results/asqa_flare

In [ ]:
# ── WikiAsp ───────────────────────────────────────────────────────
!python scripts/run_flare.py --eval_mode no_retrieval --dataset wikiasp --data_path data/wikiasp --results_dir /content/drive/MyDrive/active-rag-results/wikiasp_no_retrieval

In [ ]:
!python scripts/run_flare.py --eval_mode single_retrieval --dataset wikiasp --data_path data/wikiasp --results_dir /content/drive/MyDrive/active-rag-results/wikiasp_single_retrieval

In [ ]:
!python scripts/run_flare.py --eval_mode flare --dataset wikiasp --data_path data/wikiasp --results_dir /content/drive/MyDrive/active-rag-results/wikiasp_flare